# Jigsaw Toxic Comment Classification

## Tokenization

### Objective
Convert raw comment text into clean numerical sequences suitable for deep learning models.

### Tasks
- Mount Google Drive and load dataset
- Remove URLs, Wikipedia markup, punctuation, and uninformative stopwords
- Build a vocabulary from training data
- Convert text to integer sequences
- Pad sequences to a fixed length
- Save all artifacts to Google Drive for the BiLSTM notebook

## 1. Environment detection (Colab vs. local)

In [4]:
import os

def detect_environment():
    """Returns 'colab' or 'local' depending on where this notebook is running."""
    try:
        import google.colab  # only importable inside Colab
        return "colab"
    except ImportError:
        return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV}")


Detected environment: local


## 2. Import Libraries

we use:
- pandas and numpy for data handling
- re for text cleaning with regular expressions
- nltk for stopword list
- keras Tokenizer for vocabulary building and sequence encoding
- keras pad_sequences for fixed-length padding
- pickle to save the tokenizer

In [5]:
import pandas as pd
import numpy as np
import re
import pickle

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## 3. Load data

Paths are set automatically based on the detected environment:
- **Colab**: mounts your Google Drive and reads from a shared `jigsaw-data/` folder
  (make sure this folder has been added to your own Drive first — see the README's
  Drive setup instructions if you haven't done this yet).
- **Local**: reads directly from `../data/` relative to this notebook, same convention
  as `00_local_setup.ipynb`.

In [ ]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

    # This assumes your "jigsaw-data" folder has been added to your drive
    # If your folder is named differently, change the line below to match.
    DATA_DIR = "/content/drive/MyDrive/jigsaw-data/"
    TRAIN_PATH = DATA_DIR + "train.csv/train.csv"
    TEST_PATH = DATA_DIR + "test.csv/test.csv"
    TEST_LABELS_PATH = DATA_DIR + "test_labels.csv/test_labels.csv"
else:
    DATA_DIR = os.path.join("..", "data")
    TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
    TEST_PATH = os.path.join(DATA_DIR, "test.csv")
    TEST_LABELS_PATH = os.path.join(DATA_DIR, "test_labels.csv")

for name, path in [("train.csv", TRAIN_PATH), ("test.csv", TEST_PATH), ("test_labels.csv", TEST_LABELS_PATH)]:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{status:8s} {name:18s} -> {path}")

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(train.shape)
print(test.shape)


FOUND    train.csv          -> ..\data\train.csv
FOUND    test.csv           -> ..\data\test.csv
FOUND    test_labels.csv    -> ..\data\test_labels.csv
(159571, 8)
(153164, 2)


## 4. Custom Stopword List

Standard stopwords are filtered, but **toxic-signal words are kept**.
Words like `you`, `your`, `i`, `me` appear heavily in toxic comments
and must not be removed.

In [7]:
base_stopwords = set(stopwords.words('english'))

# keep these — they carry toxicity signal
keep = {
    'you', 'your', 'yourself', 'i', 'me', 'my', 'myself',
    'we', 'our', 'they', 'their', 'them', 'he', 'she',
    'no', 'not', 'nor', 'never', "don't", "won't", 'will'
}

STOPWORDS = base_stopwords - keep
print(f"Stopwords to remove: {len(STOPWORDS)}")

Stopwords to remove: 178


## 5. Text Cleaning

Cleaning steps applied in order:
1. Lowercase
2. Remove URLs
3. Remove Wikipedia markup (`==`, `\\n`)
4. Remove punctuation and non-alphabetic characters
5. Remove custom stopwords
6. Collapse extra whitespace

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)      # remove URLs
    text = re.sub(r'={2,}', '', text)                  # remove == markup
    text = re.sub(r'\n', ' ', text)                    # remove newlines
    text = re.sub(r'[^a-z\s]', '', text)               # keep only letters
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS] # remove stopwords
    return ' '.join(tokens)

In [9]:
train['clean_text'] = train['comment_text'].apply(clean_text)
test['clean_text']  = test['comment_text'].apply(clean_text)

train['clean_text'].head()

0    explanation edits made my username hardcore me...
1    daww he matches background colour im seemingly...
2    hey man im really not trying edit war guy cons...
3    i cant make real suggestions improvement i won...
4       you sir my hero chance you remember page thats
Name: clean_text, dtype: str

## 6. Verify Cleaning on Toxic Sample

Check that toxic signal words are preserved after cleaning.

In [10]:
sample = train[train['toxic'] == 1].iloc[0]
print(f"Original : {sample['comment_text'][:80].strip()}")
print(f"Cleaned  : {sample['clean_text'][:80]}")

Original : COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK
Cleaned  : cocksucker you piss around my work


## 7. Tokenizer Configuration

Key hyperparameters:
- `MAX_VOCAB` — maximum number of unique words to keep
- `MAX_LEN` — maximum sequence length (longer sequences are truncated)

In [11]:
MAX_VOCAB = 20000
MAX_LEN   = 200

## 8. Fit Tokenizer on Training Data

The tokenizer is fit only on training text to avoid data leakage.

In [12]:
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer.fit_on_texts(train['clean_text'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulary size: {min(vocab_size, MAX_VOCAB)}")

Vocabulary size: 20000


## 9. Inspect Vocabulary

Top tokens should now reflect meaningful content words, not stopwords.

In [13]:
top_tokens = list(tokenizer.word_index.items())[:15]
print("Top 15 tokens:")
for word, idx in top_tokens:
    print(f"  {word:<12}: {idx}")

Top 15 tokens:
  <OOV>       : 1
  you         : 2
  i           : 3
  not         : 4
  your        : 5
  article     : 6
  page        : 7
  my          : 8
  me          : 9
  wikipedia   : 10
  talk        : 11
  will        : 12
  please      : 13
  would       : 14
  no          : 15


## 10. Text to Sequences

Each word is replaced by its integer index from the vocabulary.

In [14]:
train_sequences = tokenizer.texts_to_sequences(train['clean_text'])
test_sequences  = tokenizer.texts_to_sequences(test['clean_text'])

print(f"Example sequence (first 15 tokens): {train_sequences[0][:15]}")

Example sequence (first 15 tokens): [542, 58, 61, 8, 532, 4413, 11213, 939, 225, 18, 1903, 10664, 6331, 2498, 3]


## 11. Sequence Length Distribution

After stopword removal sequences are shorter — verify MAX_LEN still covers 95%+ of data.

In [15]:
lengths = [len(s) for s in train_sequences]

print(f"Mean length    : {np.mean(lengths):.1f}")
print(f"Median length  : {np.median(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.1f}")
print(f"Max length     : {np.max(lengths)}")
print(f"MAX_LEN={MAX_LEN} covers 95%+ of comments ✓")

Mean length    : 39.4
Median length  : 21.0
95th percentile: 132.0
Max length     : 1387
MAX_LEN=200 covers 95%+ of comments ✓


## 12. Pad Sequences

All sequences are padded (or truncated) to `MAX_LEN`.
- Padding is added **post** (at the end)
- Truncation is done **post** (tail is removed)

In [16]:
X_train = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(test_sequences,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

X_train shape: (159571, 200)
X_test shape : (153164, 200)


## 13. Prepare Labels

In [17]:
labels  = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
y_train = train[labels].values

print(f"y_train shape: {y_train.shape}")

y_train shape: (159571, 6)


## 14. Save Artifacts to Google Drive

All artifacts are saved directly to Google Drive so `bilstm.ipynb` can
load them in a new session without re-running tokenization.

In [18]:
ARTIFACTS_DIR = "/content/drive/MyDrive/jigsaw-data/"

np.save(ARTIFACTS_DIR + 'X_train.npy', X_train)
np.save(ARTIFACTS_DIR + 'X_test.npy',  X_test)
np.save(ARTIFACTS_DIR + 'y_train.npy', y_train)

with open(ARTIFACTS_DIR + 'tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print(f"Saved all artifacts to: {ARTIFACTS_DIR}")
print("  X_train.npy")
print("  X_test.npy")
print("  y_train.npy")
print("  tokenizer.pkl")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/jigsaw-data/X_train.npy'

## Key Findings

- URLs and Wikipedia markup removed before tokenization.
- Stopwords filtered with a custom list that preserves toxic-signal pronouns.
- Vocabulary capped at 20,000 most frequent meaningful words.
- Sequences padded/truncated to 200 tokens — covers 95%+ of comments.
- All artifacts saved to Google Drive for direct reuse in `bilstm.ipynb`.

### Next Steps

- Load pre-trained GloVe embeddings in `bilstm.ipynb`
- Build embedding matrix aligned with tokenizer vocabulary
- Train BiLSTM model on X_train and y_train